# Optimisation avec Optax

Optax fournit des transformations de gradients pour ajuster les paramètres d'une machine. Il ne définit pas la machine et ne calcule pas lui-même le gradient : la fonction objectif et la différentiation restent du ressort de JAX.

Ce notebook suppose connus les tableaux, `jax.grad`, `jax.value_and_grad`, `jax.jit` et les arborescences de paramètres (`'PyTrees'`). Nous nous limitons à deux exemples concis : une fonction quadratique scalaire et une machine affine.

## Exercices

1. [Choix du pas pour une fonction quadratique](#Exercice-1)
2. [Comparer deux méthodes sur une machine affine](#Exercice-2)

In [1]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import optax

print(f"JAX {jax.__version__}, Optax {optax.__version__}")

JAX 0.7.0, Optax 0.2.5


## 1. Schéma d'utilisation

Pour minimiser une fonction $J(p)$ :

1. `optimiseur.init(p)` initialise l'état interne de la méthode ;
2. JAX calcule $J(p)$ et $\nabla J(p)$ ;
3. `optimiseur.update` transforme le gradient en une mise à jour ;
4. `optax.apply_updates` construit les nouveaux paramètres.

L'état de l'optimiseur contient, selon la méthode, le numéro de l'itération ou des moyennes de gradients antérieurs. Il est distinct des paramètres de la machine.

## 2. Descente de gradient sur une fonction quadratique

Considérons

$$
J(p)=\frac12(p-3)^2.
$$

La fonction `optax.sgd` représente ici la descente de gradient (`'gradient descent'`) à pas constant.

In [2]:
def cout_scalaire(p):
    return 0.5 * (p - 3.0) ** 2

alpha = 0.2
optimiseur = optax.sgd(learning_rate=alpha)
p = jnp.array(-4.0)
etat = optimiseur.init(p)

@jax.jit
def pas_scalaire(p, etat):
    valeur, gradient = jax.value_and_grad(cout_scalaire)(p)
    mise_a_jour, nouvel_etat = optimiseur.update(gradient, etat, p)
    nouveau_p = optax.apply_updates(p, mise_a_jour)
    return nouveau_p, nouvel_etat, valeur

for k in range(25):
    p, etat, valeur = pas_scalaire(p, etat)
    if k in (0, 4, 9, 24):
        print(f"{k + 1:2d} : p = {p:.8f}, J = {valeur:.3e}")

 1 : p = -2.60000000, J = 2.450e+01
 5 : p = 0.70624000, J = 4.110e+00
10 : p = 2.24838072, J = 4.414e-01
25 : p = 2.97355475, J = 5.464e-04


### Exercice 1

**Choix du pas pour une fonction quadratique.**

Pour un pas $\alpha>0$, montrez que l'erreur $e_k=p_k-3$ vérifie

$$
e_{k+1}=(1-\alpha)e_k.
$$

Déduisez la condition sur $\alpha$ assurant la convergence. Vérifiez-la numériquement avec `alpha=0.8`, `1.5`, `2.0` et `2.1`, en transmettant cette valeur à `learning_rate`.

In [ ]:
# À compléter.

## 3. Ajustement d'une machine affine

Nous reprenons la machine $x\mapsto ax+b$. Ses paramètres sont organisés dans un dictionnaire. Optax applique automatiquement la même transformation à chaque feuille de cette arborescence.

Pour illustrer un optimiseur avec état, nous utilisons la méthode d'Adam (`'Adam optimizer'`). La fonction objectif reste l'erreur quadratique moyenne.

In [5]:
def machine_affine(parametres, x):
    return parametres["a"] * x + parametres["b"]

def cout_quadratique(parametres, x, z):
    residu = machine_affine(parametres, x) - z
    return 0.5 * jnp.mean(residu**2)

x = jnp.linspace(-1.0, 1.0, 41)
z = 2.0 * x - 1.0
parametres = {"a": jnp.array(0.0), "b": jnp.array(0.0)}

In [6]:
alpha_affine = 0.1
optimiseur_affine = optax.adam(learning_rate=alpha_affine)
etat_affine = optimiseur_affine.init(parametres)

@jax.jit
def pas_affine(parametres, etat, x, z):
    valeur, gradient = jax.value_and_grad(cout_quadratique)(parametres, x, z)
    mise_a_jour, nouvel_etat = optimiseur_affine.update(
        gradient, etat, parametres
    )
    nouveaux_parametres = optax.apply_updates(parametres, mise_a_jour)
    return nouveaux_parametres, nouvel_etat, valeur

for k in range(150):
    parametres, etat_affine, valeur = pas_affine(
        parametres, etat_affine, x, z
    )

print(f"paramètres : {parametres}")
print(f"coût final : {cout_quadratique(parametres, x, z):.3e}")

paramètres : {'a': Array(2.00017334, dtype=float64), 'b': Array(-1.00013404, dtype=float64)}
coût final : 1.424e-08


### Exercice 2

**Comparer deux méthodes sur une machine affine.**

Remplacez `optax.adam` par `optax.sgd`. Cherchez un pas donnant une convergence rapide et comparez les premières valeurs du coût. Recommencez après avoir remplacé $x$ par `100 * x`. Expliquez pourquoi le choix d'un même pas devient plus délicat.

Dans chaque expérience, n'oubliez pas de réinitialiser à la fois les paramètres et l'état de l'optimiseur.

In [ ]:
# À compléter.

## Bilan

Optax sépare trois objets : les paramètres de la machine, le gradient calculé par JAX et l'état propre à la méthode d'optimisation. L'interface `init`–`update`–`apply_updates` restera identique lorsque les paramètres proviendront d'un réseau défini avec Flax.